<a href="https://colab.research.google.com/github/FernandaXdea/MachineLearning/blob/main/Projeto_de_Transfer_Learning_em_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

O projeto consiste em aplicar o método de Transfer Learning em uma rede de Deep Learning na linguagem Python no ambiente COLAB.  

Para exemplo, utilizaremos o seguinte projeto que realiza Transfer Learning com o Dataset do MNIST:
https://colab.research.google.com/github/kylemath/ml4a-guides/blob/master/notebooks/transfer-learning.ipynb

O dataset utilizado engloba duas classes: gatos e cachorros. Uma descrição da base de dados pode ser visualizada neste link: https://www.tensorflow.org/datasets/catalog/cats_vs_dogs.

Já o dataset para download pode ser acessado por meio deste outro link:

https://www.microsoft.com/en-us/download/details.aspx?id=54765.



Observações: Neste projeto, você pode usar sua própria base de dados (exemplo: fotos suas, dos seus pais, dos seus amigos, dos seus animais domésticos, etc), o exemplo de gatos e cachorros, pode ser substituído por duas outras classes do seu interesse. O Dataset criado em nosso projeto anterior, pode ser utilizado agora.  

O projeto deve ser enviado para o GitHub da DIO: https://github.com/digitalinnovationone.

In [1]:
# Importação de bibliotecas necessárias
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import zipfile
import os

# Download do dataset (gatos e cachorros)
dataset_url = "https://download.microsoft.com/download/7/E/1/7E1C81BF-C3DB-4FBF-B7D0-8C2F99D5CCEA/kagglecatsanddogs_3367a.zip"
dataset_path = tf.keras.utils.get_file("cats_and_dogs.zip", origin=dataset_url, extract=True)
dataset_dir = os.path.join(os.path.dirname(dataset_path), 'PetImages')

# Pré-processamento dos dados
batch_size = 32
img_size = (224, 224)

datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2  # 20% dos dados para validação
)

train_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

# Carregamento do modelo pré-treinado (MobileNetV2)
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

# Congelar os pesos do modelo base
base_model.trainable = False

# Adicionar camadas específicas para a classificação
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

# Construir o modelo completo
model = Model(inputs=base_model.input, outputs=predictions)

# Compilação do modelo
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Treinamento do modelo
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    steps_per_epoch=train_generator.samples // batch_size,
    validation_steps=val_generator.samples // batch_size
)

# Salvar o modelo treinado
model.save('cats_dogs_transfer_learning.h5')

# Avaliação do modelo
loss, accuracy = model.evaluate(val_generator)
print(f"Acurácia no conjunto de validação: {accuracy:.2f}")

# Teste com nova imagem
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    img = image.load_img(img_path, target_size=img_size)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array)
    if prediction[0] > 0.5:
        return "Cachorro"
    else:
        return "Gato"

# Testando com uma imagem
img_path = "caminho/para/sua/imagem.jpg"
print(predict_image(img_path))


Exception: URL fetch failure on https://download.microsoft.com/download/7/E/1/7E1C81BF-C3DB-4FBF-B7D0-8C2F99D5CCEA/kagglecatsanddogs_3367a.zip: 404 -- Not Found